# LangChain Tutorial — Student Educational Support System
## Real Implementation with OpenAI — DSS5105 Week 2 & 5

### Learning Objectives
- Master core LangChain components: Models, Prompts, Chains, Memory, Retrieval, Tools, Agents
- Build a complete educational support chatbot from start to finish
- Understand how components work together in a real application
- See how capabilities are packaged as **Skills** and executed by an **agent harness**

### What changed from the 2025 edition
This notebook builds the same educational support system, but on **LangChain 1.x**, where several 0.x helper classes were retired in favour of composable primitives.

| 2025 (LangChain 0.x) | 2026 (LangChain 1.x) | Why |
|---|---|---|
| `ConversationChain` + `ConversationBufferMemory` | the message list you carry forward, or a LangGraph **checkpointer** | memory is state, not a wrapper class |
| `RetrievalQA.from_chain_type(...)` | `retriever \| prompt \| llm \| parser` (LCEL) | one composition style for everything |
| `initialize_agent(..., AgentType.X)` | `create_agent(model, tools, ...)` | one agent builder, native tool calling |
| `Chroma` / `langchain.vectorstores` | `InMemoryVectorStore`, `FAISS` from their own packages | integrations live outside core |
| — | **Skills & harness** (§9) | how production agents package capabilities |

### Setup (uv + Python 3.12)
```bash
# from this Tutorial/ folder
uv sync --all-groups          # creates .venv and installs the locked deps
echo "OPENAI_API_KEY=sk-..." > .env
uv run jupyter lab            # then open this notebook
```

> **No API key?** Every cell that calls the API is **guarded**, so the notebook still runs top-to-bottom and simply prints a `[skipped]` note. Add a key in `.env` to see real model output.

### All Required Imports

In [1]:
import os
import json
import warnings

warnings.filterwarnings("ignore")

from dotenv import load_dotenv

# Models, prompts, parsers, messages
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Retrieval & vector stores
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Tools, agents & memory
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

print("✅ Imports loaded")

✅ Imports loaded


### Environment Setup and API Key Verification

`ChatOpenAI` reads `OPENAI_API_KEY` from the environment, so `load_dotenv()` is all the wiring you need.
Keep the model name in **one constant** — swapping models later is then a one-line change.

In [2]:
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
HAS_KEY = bool(openai_api_key)

MODEL = "gpt-5.4-mini"                    # change here to swap models
EMBED_MODEL = "text-embedding-3-small"


def skipped(what):
    """Placeholder so the notebook still runs end-to-end without an API key."""
    print(f"[skipped: no OPENAI_API_KEY] {what}")


print("✅ OpenAI API key found and loaded" if HAS_KEY
      else "⚠️  No OPENAI_API_KEY found — cells that call the API will be skipped")
print(f"Model: {MODEL} | Embeddings: {EMBED_MODEL}")

✅ OpenAI API key found and loaded
Model: gpt-5.4-mini | Embeddings: text-embedding-3-small


## 1. Models - Basic Setup
**Models are the AI brains (like GPT-5.4) that understand your questions and generate intelligent responses.**

### Understanding Message Types in LangChain

This is how LangChain keeps track of who said what in a conversation — it uses different message types for different roles:

- **HumanMessage** → input from the user (like you typing a query).
- **AIMessage** → responses generated by the AI model.
- **SystemMessage** → system-level instructions (e.g., context, rules, or prompts that guide the AI's behavior).
- **ToolMessage** → the result of a tool call, fed back to the model (you will meet this in §7-§8).

So in your code:
```python
response = llm.invoke([HumanMessage(content=test_input)])
```

You're passing the model a list of messages, where the first message is from the human (you). The model then replies with an AIMessage.

👉 If you just passed a raw string, it wouldn't know who the message was from, but with `HumanMessage`, it knows it's your input in a dialogue. (LangChain 1.x also accepts a plain string or a list of `{"role": ..., "content": ...}` dicts as a shortcut.)

In [3]:
test_input = "What are the top 3 benefits of online learning platforms?"

print("🔵 INPUT TO GPT:")
print(f"Query: {test_input}")

if HAS_KEY:
    # Initialize the model — one interface, any provider
    llm = ChatOpenAI(model=MODEL, temperature=0.1)

    print(f"\n🔵 CALLING {MODEL.upper()}...")
    response = llm.invoke([HumanMessage(content=test_input)])

    print("\n🤖 OUTPUT FROM GPT:")
    print(response.content)
else:
    skipped(f"ChatOpenAI(model='{MODEL}').invoke([HumanMessage(...)])")

🔵 INPUT TO GPT:
Query: What are the top 3 benefits of online learning platforms?

🔵 CALLING GPT-5.4-MINI...

🤖 OUTPUT FROM GPT:
The top 3 benefits of online learning platforms are:

1. **Flexibility**
   - You can learn anytime and anywhere, making it easier to fit education around work, family, or other commitments.

2. **Wide variety of courses**
   - Online platforms often offer a huge range of subjects and skill levels, from academic topics to professional certifications and hobbies.

3. **Cost-effectiveness**
   - They are often cheaper than traditional classes, and you save on travel, housing, and other related costs.

If you want, I can also give you the **top 3 drawbacks** or compare **online learning vs. in-person learning**.


## 2. Prompts and Templates
**Prompts are structured instructions with placeholders that tell the model exactly what to do with your data.**

In [4]:
# Block 1: Basic Educational Support Template
student_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an educational support specialist. Be encouraging and helpful."),
    ("human", "Analyze this student query: {student_query}"),
    ("human", "Provide urgency level (1-5) and suggested response category."),
])

# Format and use the prompt
student_query = "I can't access my course materials and have an exam tomorrow!"
formatted_prompt = student_prompt.format_messages(student_query=student_query)

print("🔵 INPUT TO GPT:")
print("Template: Educational Support Analysis")
print(f"Student Query: {student_query}")
print(f"Formatted Messages: {[msg.content for msg in formatted_prompt]}")

if HAS_KEY:
    print(f"\n🔵 CALLING {MODEL.upper()}...")
    response = llm.invoke(formatted_prompt)

    print("\n🤖 OUTPUT FROM GPT:")
    print(response.content)
else:
    skipped("llm.invoke(formatted_prompt)")

🔵 INPUT TO GPT:
Template: Educational Support Analysis
Student Query: I can't access my course materials and have an exam tomorrow!
Formatted Messages: ['You are an educational support specialist. Be encouraging and helpful.', "Analyze this student query: I can't access my course materials and have an exam tomorrow!", 'Provide urgency level (1-5) and suggested response category.']

🔵 CALLING GPT-5.4-MINI...

🤖 OUTPUT FROM GPT:
**Urgency level:** **5/5**  
This is very urgent because the student cannot access course materials and has an exam tomorrow, which directly affects immediate academic performance.

**Suggested response category:** **Technical/access issue with time-sensitive academic impact**  
A good response should prioritize:
- restoring access quickly,
- offering immediate troubleshooting steps,
- directing the student to urgent support (IT/help desk/instructor/course admin),
- and asking for key details like the platform, error message, and course name.


In [5]:
# Block 2: Multi-role Conversation Template
support_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role} at {platform}. Use {tone} tone."),
    ("human", "Student issue: {issue}"),
    ("human", "Previous context: {context}"),
    ("human", "Generate a response that addresses the issue and follows up appropriately."),
])

response_inputs = {
    "role": "Senior Academic Advisor",
    "platform": "EduTech Learning Platform",
    "tone": "supportive and encouraging",
    "issue": "Struggling with advanced calculus concepts",
    "context": "Student is in their second year, enrolled in premium tutoring",
}

formatted_chat = support_template.format_messages(**response_inputs)

print("🔵 INPUT TO GPT:")
print("Template: Multi-role Support Response")
print(f"Variables: {response_inputs}")

if HAS_KEY:
    print(f"\n🔵 CALLING {MODEL.upper()}...")
    response = llm.invoke(formatted_chat)

    print("\n🤖 OUTPUT FROM GPT:")
    print(response.content[:200] + "...")
else:
    skipped("llm.invoke(formatted_chat)")

🔵 INPUT TO GPT:
Template: Multi-role Support Response
Variables: {'role': 'Senior Academic Advisor', 'platform': 'EduTech Learning Platform', 'tone': 'supportive and encouraging', 'issue': 'Struggling with advanced calculus concepts', 'context': 'Student is in their second year, enrolled in premium tutoring'}

🔵 CALLING GPT-5.4-MINI...

🤖 OUTPUT FROM GPT:
I’m sorry to hear advanced calculus has been feeling overwhelming. Since you’re in your second year and enrolled in premium tutoring, you do have strong support available, and we can use it to make th...


## 3. Chains - Connecting Components
**Chains link multiple steps together (prompt → model → parser) to create reusable AI workflows.**

### Understanding the Pipe Operator (|) in LangChain

#### What does | mean here?

In LangChain's **Expression Language (LCEL)**, the `|` is operator overloading for pipe composition.

It works a lot like a Unix shell pipeline (`cmd1 | cmd2 | cmd3`), but instead of passing raw text, each component in LangChain has a well-defined input/output schema. The pipe (`|`) just wires them together.

So this:
```python
prompt | llm | StrOutputParser()
```

means:
1. Take the `ChatPromptTemplate` (`prompt`)
2. Format it with the given inputs → produces messages
3. Send those to `llm` (the model) → produces an `AIMessage`
4. Pass the response into `StrOutputParser()` → returns a clean string

Each `|` passes the output of the left component as input to the right component, creating a seamless data flow pipeline.

#### Chain vs Manual Steps Comparison

**❌ Manual Approach (3 separate steps):**
```python
formatted_prompt = prompt.format_messages(metric=metric_input)   # 1. format
raw_response = llm.invoke(formatted_prompt)                      # 2. call the model
result = parser.invoke(raw_response)                             # 3. parse
```

**✅ Chain Approach (1 simple step):**
```python
analysis_chain = prompt | llm | parser
result = analysis_chain.invoke({"metric": metric_input})
```

**Benefits of Chains:**
- **Less Code**: 1 line instead of 3 separate operations
- **No Intermediate Variables**: No need to manage `formatted_prompt` or `raw_response`
- **Automatic Data Flow**: Each component's output automatically becomes the next component's input
- **Reusable Pipeline**: Create once, use multiple times with different inputs
- **Free extras**: every chain also gets `.batch()`, `.stream()` and `.ainvoke()` for nothing

In [6]:
# Create educational data analysis chain
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an educational data analyst. Be concise."),
    ("human", "Analyze this learning metric: {metric}"),
])

parser = StrOutputParser()

metric_input = "Course completion rate dropped by 20%"

print("🔵 INPUT TO GPT:")
print("Chain: Educational Data Analysis")
print(f"Metric: {metric_input}")

if HAS_KEY:
    # Create chain using the pipe operator
    analysis_chain = prompt | llm | parser

    print(f"\n🔵 CALLING {MODEL.upper()}...")
    result = analysis_chain.invoke({"metric": metric_input})

    print("\n🤖 OUTPUT FROM GPT:")
    print(result)
else:
    skipped("(prompt | llm | parser).invoke({'metric': ...})")

🔵 INPUT TO GPT:
Chain: Educational Data Analysis
Metric: Course completion rate dropped by 20%

🔵 CALLING GPT-5.4-MINI...

🤖 OUTPUT FROM GPT:
A **20% drop in course completion rate** suggests learners are starting the course but fewer are finishing it.

### What it may indicate
- **Engagement decline:** Content may be less engaging or too long.
- **Difficulty spike:** A module may be too hard or confusing.
- **Technical/friction issues:** Access problems, broken links, slow loading, or poor mobile experience.
- **Expectation mismatch:** Learners may realize the course doesn’t match their goals.
- **External factors:** Seasonality, workload, or cohort differences.

### What to check next
1. **Where drop-off happens**  
   Identify the lesson/module where most learners exit.
2. **Segment the data**  
   Compare by device, learner group, start date, instructor, or course version.
3. **Engagement signals**  
   Look at time spent, quiz attempts, video completion, and activity completion.
4

## 4. Memory - Conversation Context
**Memory stores conversation history so the AI remembers what was discussed earlier in the chat.**

### How memory works in LangChain 1.x

Models are **stateless** — each call knows only what you send it. "Memory" is therefore just *the message list you carry forward*. The 0.x memory classes were wrappers around that idea, and they are gone:

| 2025 memory class | What it did | 2026 equivalent |
|---|---|---|
| `ConversationBufferMemory` | keep every turn | keep appending to the message list |
| `ConversationBufferWindowMemory` | keep the last *k* turns | slice the list, or `trim_messages(...)` |
| `ConversationSummaryBufferMemory` | summarise older turns | `trim_messages` + a summarisation step |
| `ConversationChain` | glue llm + memory | `llm.invoke(messages)` or `create_agent(..., checkpointer=...)` |

Two things grow with the conversation: **cost** and **latency** — every turn resends the whole history. That is why trimming and summarising exist.

In [7]:
# Manual memory: keep the message list and pass it back every turn
messages = [SystemMessage(content="You are a helpful academic advisor. Keep replies short.")]

input1 = "Hi, I'm Alice and I'm struggling with my CS101 course"
input2 = "What's my name and what course am I taking?"

if HAS_KEY:
    def say(text):
        """One conversational turn: append the human message, call, append the reply."""
        messages.append(HumanMessage(content=text))
        reply = llm.invoke(messages)          # the WHOLE history goes to the model
        messages.append(reply)
        return reply.content

    print("🔵 INPUT TO GPT:")
    print("Conversation with Memory - Turn 1")
    print(f"Student: {input1}")
    print(f"\n🔵 CALLING {MODEL.upper()}...")
    print("\n🤖 OUTPUT FROM GPT:")
    print("AI:", say(input1))

    print("\n🔵 INPUT TO GPT:")
    print("Conversation with Memory - Turn 2")
    print(f"Student: {input2}")
    print(f"\n🔵 CALLING {MODEL.upper()}...")
    print("\n🤖 OUTPUT FROM GPT:")
    print("AI:", say(input2))

    print("\n📝 Memory Buffer:")
    for m in messages:
        print(f"  {m.type:>9}: {m.content[:70]}")
else:
    skipped("two turns where the full message list is resent each time")

🔵 INPUT TO GPT:
Conversation with Memory - Turn 1
Student: Hi, I'm Alice and I'm struggling with my CS101 course

🔵 CALLING GPT-5.4-MINI...

🤖 OUTPUT FROM GPT:
AI: Hi Alice — I’m sorry you’re struggling. I can help with CS101.

What part is giving you trouble?
- programming basics
- loops/conditionals
- functions
- arrays/lists
- debugging
- homework/project prep
- exam studying

If you want, paste a problem or code snippet and I’ll walk through it with you.

🔵 INPUT TO GPT:
Conversation with Memory - Turn 2
Student: What's my name and what course am I taking?

🔵 CALLING GPT-5.4-MINI...

🤖 OUTPUT FROM GPT:
AI: Your name is Alice, and you’re taking CS101.

📝 Memory Buffer:
     system: You are a helpful academic advisor. Keep replies short.
      human: Hi, I'm Alice and I'm struggling with my CS101 course
         ai: Hi Alice — I’m sorry you’re struggling. I can help with CS101.

What p
      human: What's my name and what course am I taking?
         ai: Your name is Alice, and you’r

### Production memory: a checkpointer

Carrying a Python list works inside one notebook. For a real app you want history **persisted per user/session**, which is what a LangGraph **checkpointer** does: pass `checkpointer=` to `create_agent`, then identify each conversation with a `thread_id`. Same `thread_id` → same memory; new `thread_id` → fresh start.

Swap `InMemorySaver()` for a Postgres/SQLite saver and the conversation survives a restart.

In [8]:
if HAS_KEY:
    remembering_agent = create_agent(
        model=MODEL,
        tools=[],                                    # no tools needed for plain chat
        system_prompt="You are a helpful academic advisor. Keep replies short.",
        checkpointer=InMemorySaver(),                # <- this is the memory
    )

    session = {"configurable": {"thread_id": "alice-001"}}   # one thread = one conversation

    for turn in ["Hi, I'm Alice and I'm taking CS101.", "Which course did I say I'm taking?"]:
        out = remembering_agent.invoke({"messages": [{"role": "user", "content": turn}]}, config=session)
        print(f"Student: {turn}")
        print(f"AI: {out['messages'][-1].content}\n")

    fresh = {"configurable": {"thread_id": "someone-else"}}   # different thread = no memory
    out = remembering_agent.invoke({"messages": [{"role": "user", "content": "Which course did I say I'm taking?"}]},
                                   config=fresh)
    print("New thread (no shared memory):", out["messages"][-1].content)
else:
    skipped("create_agent(..., checkpointer=InMemorySaver()) keyed by thread_id")

Student: Hi, I'm Alice and I'm taking CS101.
AI: Hi Alice — nice to meet you! What can I help with in CS101?

Student: Which course did I say I'm taking?
AI: You said you’re taking **CS101**.

New thread (no shared memory): I don’t have that information in this chat. If you tell me the course name, I can remember it for the rest of our conversation.


## 5. Retrieval - Knowledge Base Integration
**Retrieval searches through documents/databases to find relevant information before answering questions.**

The recipe is always the same: **embed** your documents into a vector store → **retrieve** the closest chunks for a question → **stuff** them into the prompt as context → let the model answer *from that context only*.

In [9]:
# Sample educational policies for knowledge base
educational_policies = [
    "Course Drop Policy: Students can drop courses within 2 weeks for full refund. Partial refunds available until week 4.",
    "Grade Appeal Process: Students must submit grade appeals within 2 weeks of grade posting. Appeals are reviewed by academic committee.",
    "Academic Probation: Students with GPA below 2.0 for two consecutive semesters are placed on academic probation.",
    "Extension Policy: Course extensions up to 30 days require instructor approval. Extensions over 30 days need academic advisor approval.",
    "Technical Support: All students receive email support within 24 hours. Priority support available for premium students.",
    "Graduation Requirements: Students must complete 120 credits with minimum 2.0 GPA and all core course requirements.",
]


def format_docs(docs):
    """Turn retrieved Documents into one context string for the prompt."""
    return "\n\n".join(d.page_content for d in docs)


def build_qa_chain(retriever, temperature=0):
    """retriever + prompt + llm + parser — the LCEL replacement for RetrievalQA."""
    qa_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an educational support specialist. Answer using ONLY this context:\n\n{context}"),
        ("human", "{question}"),
    ])
    return (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | qa_prompt
        | ChatOpenAI(model=MODEL, temperature=temperature)
        | StrOutputParser()
    )


if HAS_KEY:
    print("📚 Creating Knowledge Base...")
    policy_store = InMemoryVectorStore.from_texts(
        educational_policies,
        embedding=OpenAIEmbeddings(model=EMBED_MODEL),
    )
    policy_retriever = policy_store.as_retriever(search_kwargs={"k": 2})
    policy_qa = build_qa_chain(policy_retriever)
    print(f"✅ Embedded {len(educational_policies)} policies and built the QA chain")
else:
    skipped("embedding the policies into InMemoryVectorStore")

📚 Creating Knowledge Base...
✅ Embedded 6 policies and built the QA chain


### Understanding the retrieval chain

**🔍 Vector store vs retriever — why convert with `as_retriever()`?**

- **Vector store** (`InMemoryVectorStore`, `FAISS`, …) = the database that stores and searches embeddings. Methods: `.similarity_search()`, `.add_texts()`.
- **Retriever** = the standard *interface* that returns documents for a query, so it can be piped into a chain. Methods: `.invoke()`, `.batch()`.

Any retrieval system (vector, keyword, hybrid, a web search API) can wear the retriever interface — that is what makes it swappable inside LCEL.

```python
retriever = vectorstore.as_retriever(
    search_type="similarity",     # or "mmr" for max marginal relevance
    search_kwargs={"k": 2},       # return the top 2 chunks
)
```

**🗂️ Where did `chain_type="stuff"` go?**

In 2025 you chose a strategy through `RetrievalQA.from_chain_type(chain_type=...)`. In 1.x you just *write* the strategy — which is why `format_docs` above is only one line:

| Strategy | How it works | Best for | How you build it in 1.x |
|---|---|---|---|
| **stuff** | concatenate all retrieved docs into one prompt | short docs, simple questions | `retriever \| format_docs` (what we did) |
| **map_reduce** | summarise each doc, then combine summaries | many/long docs | a per-doc chain + a combine chain |
| **refine** | build the answer doc by doc, refining each time | sequential reasoning | a loop or a LangGraph graph |
| **map_rerank** | score docs, answer from the best one | one authoritative source | add a reranker to the retriever |

**"stuff" remains the default choice** — fastest, one model call, and good enough whenever the retrieved context fits the window.

In [10]:
# Test retrieval
question = "What is the course drop policy?"

print("🔵 INPUT TO RETRIEVAL QA:")
print(f"Question: {question}")
print("Action: Search knowledge base + Generate answer")

if HAS_KEY:
    print(f"\n🔵 SEARCHING KNOWLEDGE BASE AND CALLING {MODEL.upper()}...")
    sources = policy_retriever.invoke(question)      # what the chain retrieves internally
    answer = policy_qa.invoke(question)

    print("\n🤖 OUTPUT FROM GPT (with retrieved context):")
    print(f"Answer: {answer}")
    print(f"Sources: {[d.page_content[:100] + '...' for d in sources]}")
else:
    skipped("policy_qa.invoke(question)")

🔵 INPUT TO RETRIEVAL QA:
Question: What is the course drop policy?
Action: Search knowledge base + Generate answer

🔵 SEARCHING KNOWLEDGE BASE AND CALLING GPT-5.4-MINI...

🤖 OUTPUT FROM GPT (with retrieved context):
Answer: Students can drop courses within 2 weeks for a full refund. Partial refunds are available until week 4.
Sources: ['Course Drop Policy: Students can drop courses within 2 weeks for full refund. Partial refunds availa...', 'Extension Policy: Course extensions up to 30 days require instructor approval. Extensions over 30 da...']


## 6. PDF Processing with a Vector DB - Document Q&A
**PDF Processing extracts text from PDFs and creates searchable vector databases for intelligent Q&A.**

Same recipe as §5, with two extra steps at the front: **load** the PDF page by page, then **split** it into overlapping chunks so each chunk is small enough to embed and specific enough to retrieve. We persist the index with **FAISS** (`save_local`) so you only pay for embeddings once.

In [11]:
def load_and_process_pdf(pdf_path):
    """Load a PDF and split it into retrievable chunks."""
    print("📚 Loading PDF document...")
    print(f"File: {pdf_path}")

    pages = PyPDFLoader(pdf_path).load()             # one Document per page
    print(f"✅ Loaded {len(pages)} pages from PDF")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,       # characters per chunk
        chunk_overlap=200,     # overlap keeps sentences from being cut in half
        length_function=len,
    )
    docs = splitter.split_documents(pages)
    print(f"📝 Split into {len(docs)} text chunks")
    return docs


def build_pdf_index(docs, db_path="./pdf_knowledge_base"):
    """Embed the chunks into FAISS and persist the index to disk."""
    print("🔍 Creating vector embeddings...")
    store = FAISS.from_documents(docs, OpenAIEmbeddings(model=EMBED_MODEL))
    store.save_local(db_path)
    print(f"💾 Vector database saved to: {db_path}")
    # Reload it to prove persistence works (trusted local file, hence the flag)
    return FAISS.load_local(db_path, OpenAIEmbeddings(model=EMBED_MODEL),
                            allow_dangerous_deserialization=True)


pdf_file_path = "educational_course_handbook.pdf"

print("🚀 Starting PDF Processing Pipeline...")
if not os.path.exists(pdf_file_path):
    print(f"❌ PDF not found: {pdf_file_path}")
elif not HAS_KEY:
    load_and_process_pdf(pdf_file_path)          # loading + splitting needs no API key
    skipped("embedding the chunks into FAISS and answering questions")
else:
    pdf_chunks = load_and_process_pdf(pdf_file_path)
    pdf_store = build_pdf_index(pdf_chunks)
    pdf_retriever = pdf_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})
    pdf_qa = build_qa_chain(pdf_retriever)
    print("✅ PDF Q&A system created successfully")

    print(f"\n{'='*60}")
    print("TESTING PDF Q&A SYSTEM")
    print(f"{'='*60}")

    pdf_questions = [
        "What is the attendance policy?",
        "How do I appeal a grade?",
        "What academic support services are available?",
    ]

    for q in pdf_questions:
        print("\n🔵 INPUT TO PDF Q&A SYSTEM:")
        print(f"Question: {q}")
        print(f"\n🔵 SEARCHING PDF DATABASE AND CALLING {MODEL.upper()}...")

        sources = pdf_retriever.invoke(q)
        answer = pdf_qa.invoke(q)

        print("\n🤖 OUTPUT FROM GPT (with PDF context):")
        print(f"Answer: {answer}")
        print(f"📄 Sources: page {sources[0].metadata.get('page', 'Unknown')} of the PDF")
        print(f"📝 Source Text Preview: {sources[0].page_content[:150]}...")

🚀 Starting PDF Processing Pipeline...
📚 Loading PDF document...
File: educational_course_handbook.pdf
✅ Loaded 2 pages from PDF
📝 Split into 5 text chunks
🔍 Creating vector embeddings...
💾 Vector database saved to: ./pdf_knowledge_base
✅ PDF Q&A system created successfully

TESTING PDF Q&A SYSTEM

🔵 INPUT TO PDF Q&A SYSTEM:
Question: What is the attendance policy?

🔵 SEARCHING PDF DATABASE AND CALLING GPT-5.4-MINI...

🤖 OUTPUT FROM GPT (with PDF context):
Answer: Regular attendance is expected in all courses. Students missing more than 25% of classes may be administratively withdrawn from the course.
📄 Sources: page 0 of the PDF
📝 Source Text Preview: EduTech University - Course Handbook 2024
Academic Policies
Course Enrollment: Students must enroll in courses during the designated registration peri...

🔵 INPUT TO PDF Q&A SYSTEM:
Question: How do I appeal a grade?

🔵 SEARCHING PDF DATABASE AND CALLING GPT-5.4-MINI...

🤖 OUTPUT FROM GPT (with PDF context):
Answer: To appeal a grade, you

## 7. Tools - Function Calling
**Tools are custom functions the AI can call to perform specific actions like calculations, lookups, or API calls.**

The `@tool` decorator turns a plain Python function into something a model can call. Two parts of your function become the model's instructions, so write them for the model:
- the **docstring** → the tool description (this is *how the model decides* to call it)
- the **type hints** → the argument schema (this is *how the arguments get validated*)

In [12]:
@tool
def get_student_info(student_id: str) -> str:
    """Get student information from the learning management system."""
    students = {
        "STU001": {
            "name": "Alice Johnson",
            "email": "alice.johnson@university.edu",
            "major": "Computer Science",
            "gpa": 3.7,
            "current_courses": ["CS101", "MATH201", "PHYS101"],
            "support_level": "Premium",
        }
    }
    return json.dumps(students.get(student_id, {"error": "Student not found"}), indent=2)


@tool
def calculate_grade(points_earned: float, total_points: float) -> str:
    """Calculate the percentage grade and letter grade for a student."""
    if total_points == 0:
        return "Error: Total points cannot be zero"

    percentage = (points_earned / total_points) * 100
    letter = next(l for cutoff, l in [(90, "A"), (80, "B"), (70, "C"), (60, "D"), (0, "F")]
                  if percentage >= cutoff)
    return f"Grade: {percentage:.1f}% ({letter})"


student_tools = [get_student_info, calculate_grade]

print("🔧 Registered tools (name + the description the model sees):")
for t in student_tools:
    print(f"  - {t.name}: {t.description}")

# Tools are runnables — you can call them directly, no model involved
print("\n🔵 INPUT TO TOOL:")
print("Tool: Grade Calculator | Input: 85 points out of 100")
print(f"🤖 TOOL OUTPUT: {calculate_grade.invoke({'points_earned': 85, 'total_points': 100})}")

print("\n🔵 INPUT TO TOOL:")
print("Tool: Student Lookup | Input: STU001")
print(f"🤖 TOOL OUTPUT:\n{get_student_info.invoke({'student_id': 'STU001'})}")

🔧 Registered tools (name + the description the model sees):
  - get_student_info: Get student information from the learning management system.
  - calculate_grade: Calculate the percentage grade and letter grade for a student.

🔵 INPUT TO TOOL:
Tool: Grade Calculator | Input: 85 points out of 100
🤖 TOOL OUTPUT: Grade: 85.0% (B)

🔵 INPUT TO TOOL:
Tool: Student Lookup | Input: STU001
🤖 TOOL OUTPUT:
{
  "name": "Alice Johnson",
  "email": "alice.johnson@university.edu",
  "major": "Computer Science",
  "gpa": 3.7,
  "current_courses": [
    "CS101",
    "MATH201",
    "PHYS101"
  ],
  "support_level": "Premium"
}


## 8. Agents - Autonomous Decision Making
**Agents autonomously decide which tools to use and in what order to accomplish complex tasks.**

A chain is a **fixed** path (`prompt → llm → parser`). An agent runs a **loop** and picks the path at runtime:

**Reason** ("what do I need?") → **Act** (call a tool) → **Observe** (read the result) → repeat until it can answer. That loop is called **ReAct**.

### Where the 2025 `AgentType` options went

`initialize_agent(..., agent=AgentType.X)` is gone. Modern models call tools natively, so the six agent types collapse into one builder plus a few options:

| 2025 `AgentType` | What it did | 2026 equivalent |
|---|---|---|
| `ZERO_SHOT_REACT_DESCRIPTION` | ReAct reasoning in free text | `create_agent(...)` — ReAct *is* the default loop |
| `CHAT_ZERO_SHOT_REACT_DESCRIPTION` | the same, for chat models | same |
| `STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION` | JSON-shaped tool calls | native tool calling, built in |
| `OPENAI_FUNCTIONS` | provider function calling | native tool calling, built in |
| `CONVERSATIONAL_REACT_DESCRIPTION` | ReAct + memory | `create_agent(..., checkpointer=InMemorySaver())` |
| `SELF_ASK_WITH_SEARCH` | ask sub-questions, then search | say so in the `system_prompt`, or build a LangGraph graph |

**Use agents when** steps depend on earlier results. **Stick to chains when** the workflow is known — chains are cheaper, faster and predictable.

In [13]:
query = "Look up student STU001 and calculate their grade if they got 87 points out of 95 total"

print("🔵 INPUT TO AGENT:")
print("Complex Query:", query)
print("Available Tools:", [t.name for t in student_tools])

if HAS_KEY:
    agent = create_agent(
        model=MODEL,
        tools=student_tools,
        system_prompt="You are a student support assistant. Use the tools when they help.",
    )

    print(f"\n🔵 AGENT REASONING AND {MODEL.upper()} CALLS...")
    print("(the agent will make multiple calls to the model and the tools)")
    result = agent.invoke({"messages": [{"role": "user", "content": query}]})

    print("\n🤖 FINAL OUTPUT FROM AGENT:")
    print(result["messages"][-1].content)
else:
    skipped("create_agent(model=MODEL, tools=student_tools).invoke(...)")

🔵 INPUT TO AGENT:
Complex Query: Look up student STU001 and calculate their grade if they got 87 points out of 95 total
Available Tools: ['get_student_info', 'calculate_grade']

🔵 AGENT REASONING AND GPT-5.4-MINI CALLS...
(the agent will make multiple calls to the model and the tools)

🤖 FINAL OUTPUT FROM AGENT:
Student STU001 is Alice Johnson, a Computer Science major.

Grade for 87 out of 95:
- 91.6%
- A


### Watching the loop

`agent.invoke()` hides the reasoning. Streaming the intermediate messages shows the actual **Reason → Act → Observe** cycle: an AI message requesting tool calls, tool messages carrying the results, then the final answer.

In [14]:
if HAS_KEY:
    for step in agent.stream(
        {"messages": [{"role": "user", "content": "Is STU001 doing well? They scored 42 out of 60 on the midterm."}]},
        stream_mode="values",
    ):
        step["messages"][-1].pretty_print()
else:
    skipped("agent.stream(...) showing the Reason → Act → Observe loop")

================================ Human Message =================================

Is STU001 doing well? They scored 42 out of 60 on the midterm.
================================== Ai Message ==================================
Tool Calls:
  get_student_info (call_Noo1mXgTBdZunMYNcyWjdvMe)
 Call ID: call_Noo1mXgTBdZunMYNcyWjdvMe
  Args:
    student_id: STU001
================================= Tool Message =================================
Name: get_student_info

{
  "name": "Alice Johnson",
  "email": "alice.johnson@university.edu",
  "major": "Computer Science",
  "gpa": 3.7,
  "current_courses": [
    "CS101",
    "MATH201",
    "PHYS101"
  ],
  "support_level": "Premium"
}
================================== Ai Message ==================================
Tool Calls:
  calculate_grade (call_Bz879WJgAelp7cDqYphYyZq4)
 Call ID: call_Bz879WJgAelp7cDqYphYyZq4
  Args:
    points_earned: 42
    total_points: 60
================================= Tool Message =================================
Na

## 9. Skills & the Agent Harness

You just built **tools** (§7) and an **agent** (§8). Two production ideas sit on top of those:

- **Harness** = the *runtime loop* that runs the model + tools (assemble context → call model → run tool → feed result back → repeat). `create_agent` **is** a harness.
- **Skill** = a *packaged capability* = a focused system prompt + the tools it needs (+ examples), bundled and loaded together.

| Concept | What it is | Scope | Example |
|---|---|---|---|
| **Tool** | one function the model can call | one action | `calculate_grade(...)` |
| **Skill** | instructions + the tools that capability needs | one job | "answer weather questions" |
| **Harness** | the loop that runs model, tools and skills | the whole agent | `create_agent(...)` |

> A *tool* is a verb, a *skill* is a recipe, a *harness* is the kitchen.

Below we (1) **bundle a skill**, (2) load one from a **`SKILL.md`** file, (3) **route** between several skills, and (4) hand-roll a tiny harness to demystify `create_agent`.

The skills here use two deliberately tiny tools so the skill files stay self-contained and portable — the exact same pattern wraps the student tools from §7.

In [15]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"Weather in {city}: sunny, 28C."


@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


demo_tools = [get_weather, add]


# A 'skill' = focused prompt + only the tools this capability needs, bundled together
def make_weather_skill(model):
    """Skill: answer weather questions. Bundles instructions + the weather tool."""
    return create_agent(
        model=model,
        tools=[get_weather],                       # only the tool this skill needs
        system_prompt=(
            "You are a weather assistant. For any location question, call get_weather "
            "and reply in one friendly sentence. If asked anything non-weather, say you only do weather."
        ),
    )


if HAS_KEY:
    weather_skill = make_weather_skill(MODEL)
    out = weather_skill.invoke({"messages": [{"role": "user", "content": "Do I need an umbrella in London?"}]})
    print("🤖", out["messages"][-1].content)
else:
    skipped("make_weather_skill(MODEL) bundles a prompt + get_weather into a reusable skill")

🤖 In London it’s sunny and 28°C, so you probably don’t need an umbrella right now.


### The popular way: a `SKILL.md` file

In practice, skills are shipped as a **folder with a `SKILL.md`** rather than hardcoded in Python. `SKILL.md` has YAML *frontmatter* (`name`, `description` — used to decide *when* to load the skill) followed by the instructions (and, optionally, bundled scripts/resources). This is the same format used by Claude/Agent Skills and is portable across harnesses.

```
skills/
├── weather/SKILL.md       # frontmatter (name, description) + instructions
└── calculator/SKILL.md
```

The loader below reads `skills/weather/SKILL.md` and builds the same harness — but now the capability lives in a file anyone can reuse or edit without touching code.

In [16]:
from pathlib import Path


def load_skill(skill_dir, model, tools):
    """Load an Agent-Skill folder: SKILL.md (frontmatter + instructions) -> a harness."""
    text = Path(skill_dir, "SKILL.md").read_text(encoding="utf-8")
    _, frontmatter, body = text.split("---", 2)          # YAML sits between the first two '---'
    meta = {k.strip(): v.strip()
            for k, v in (l.split(":", 1) for l in frontmatter.strip().splitlines() if ":" in l)}
    print("Loaded skill:", meta.get("name"), "—", meta.get("description"))
    return create_agent(model=model, tools=tools, system_prompt=body.strip())


print(Path("skills/weather/SKILL.md").read_text(encoding="utf-8"))
print("-" * 60)

if HAS_KEY:
    weather_skill = load_skill("skills/weather", MODEL, [get_weather])
    out = weather_skill.invoke({"messages": [{"role": "user", "content": "Do I need an umbrella in London?"}]})
    print("🤖", out["messages"][-1].content)
else:
    skipped("load_skill('skills/weather', ...) reads SKILL.md and builds the harness from it")

---
name: weather-assistant
description: Answer weather questions for a location. Use whenever the user asks about current weather, temperature, or whether to bring an umbrella.
---

# Weather Assistant Skill

## Instructions

You are a weather assistant. For any location question:

1. Call the `get_weather` tool with the city name.
2. Reply in **one friendly sentence** based on the result.
3. If asked anything non-weather, say you only handle weather.

## Tools

- `get_weather(city)` — returns the current weather for a city.

## Examples

- User: "Do I need an umbrella in London?" → call `get_weather("London")`, then advise based on the result.
- User: "What's the capital of France?" → reply that you only handle weather questions.

------------------------------------------------------------
Loaded skill: weather-assistant — Answer weather questions for a location. Use whenever the user asks about current weather, temperature, or whether to bring an umbrella.
🤖 It’s sunny and 28°C in 

### Why the `description` matters: routing between skills

With **one** skill you can just load it. With **many**, the harness has to *pick the right one* — and it does that by reading each skill's **`description`**. The full instructions stay unloaded until a skill is chosen; that is **progressive disclosure**, and it is what keeps the context small as your skill library grows.

Below, `route()` chooses a skill for each request **using only the descriptions** — with the model when a key is present (the production pattern), or a keyword fallback offline.

In [17]:
def discover_skills(root="skills"):
    """Scan a skills/ dir; read each SKILL.md's frontmatter (name, description) + body."""
    skills = []
    for md_path in sorted(Path(root).glob("*/SKILL.md")):
        _, fm, body = md_path.read_text(encoding="utf-8").split("---", 2)
        meta = {k.strip(): v.strip()
                for k, v in (l.split(":", 1) for l in fm.strip().splitlines() if ":" in l)}
        skills.append({"name": meta.get("name", ""), "description": meta.get("description", ""),
                       "body": body.strip()})
    return skills


SKILL_TOOLS = {"weather-assistant": [get_weather], "calculator": [add]}   # tools each skill needs


def route(query, skills):
    """Pick the best skill for a query USING ONLY each skill's description."""
    catalog = "\n".join(f"- {s['name']}: {s['description']}" for s in skills)
    if HAS_KEY:                                       # production pattern: let the model choose
        ask = f"Skills:\n{catalog}\n\nRequest: {query}\nReply with ONLY the best skill's name."
        pick = ChatOpenAI(model=MODEL, temperature=0).invoke(ask).content.strip()
        return next((s for s in skills if s["name"] == pick), skills[0])
    stop = set("a an the to or for do i in of you your is are use when whenever about please and need these this what".split())
    qw = {w.strip("?.,") for w in query.lower().split()} - stop
    return max(skills, key=lambda s: len(qw & set(s["description"].lower().replace(".", "").split())))


skills = discover_skills("skills")
print("Discovered skills:", [s["name"] for s in skills], "\n")

for q in ["Do I need an umbrella in London?", "Please add the numbers 19 and 23"]:
    picked = route(q, skills)
    print(f"Query: {q}\n  -> routed to: {picked['name']}")
    if HAS_KEY:
        skill_agent = create_agent(model=MODEL, tools=SKILL_TOOLS[picked["name"]],
                                   system_prompt=picked["body"])
        out = skill_agent.invoke({"messages": [{"role": "user", "content": q}]})
        print("  answer:", out["messages"][-1].content)
    print()

Discovered skills: ['calculator', 'weather-assistant'] 

Query: Do I need an umbrella in London?
  -> routed to: weather-assistant
  answer: No umbrella needed in London right now—it’s sunny and 28°C.

Query: Please add the numbers 19 and 23
  -> routed to: calculator
  answer: 19 + 23 = 42.



### What `create_agent` does under the hood — a tiny harness

The loop below is the *whole idea*: call the model, run any tools it asks for, feed the results back, repeat until it stops asking. `create_agent` automates exactly this — plus memory, retries, streaming and safety limits.

In [18]:
if HAS_KEY:
    llm_with_tools = ChatOpenAI(model=MODEL, temperature=0).bind_tools(demo_tools)
    tool_by_name = {t.name: t for t in demo_tools}

    msgs = [HumanMessage("Weather in Paris, and what is 12 + 30?")]
    for _ in range(5):                                  # max steps = a stop condition
        ai = llm_with_tools.invoke(msgs)                # MODEL: think
        msgs.append(ai)
        if not ai.tool_calls:                           # no tool wanted -> done
            print("FINAL:", ai.content)
            break
        for call in ai.tool_calls:                      # ACT + OBSERVE
            result = tool_by_name[call["name"]].invoke(call["args"])
            msgs.append(ToolMessage(str(result), tool_call_id=call["id"]))
            print(f"  ran {call['name']}({call['args']}) -> {result}")
else:
    skipped("the loop: call model -> run tools -> feed results back, until the model is done")

  ran get_weather({'city': 'Paris'}) -> Weather in Paris: sunny, 28C.
  ran add({'a': 12, 'b': 30}) -> 42.0
FINAL: Weather in Paris: sunny, 28°C.

12 + 30 = 42.


## 10. Complete Educational Support System
**Combines all components (models, prompts, chains, memory, retrieval, tools, agents) into a complete working system.**

### EducationalSupportBot Architecture

#### Initialization components
- **LLM**: the chat model used for general conversation
- **Knowledge base**: `InMemoryVectorStore` of educational policies + an LCEL QA chain (§5)
- **Agent**: `create_agent` over `get_student_info` and `calculate_grade`, with a checkpointer for memory (§8)
- **History**: the message list that carries general conversation forward (§4)

#### Smart query routing
`process_query()` uses keyword-based intent detection — the cheapest possible router, and a fine starting point:

1. **Policy keywords** (`policy, drop, appeal, probation, requirement`) → **RAG chain** — search the policies, answer from them
2. **Student/grade keywords** (`student, lookup, info, grade, calculate`) → **agent with tools** — look up records, compute grades
3. **Everything else** → **conversation with memory** — general academic guidance

> In production you would replace step 1-3 with a model-based router (exactly the `route()` you wrote in §9) or a supervisor agent.

In [19]:
class EducationalSupportBot:
    """Routes a student query to retrieval, tools, or plain conversation."""

    def __init__(self, policies, tools, model=MODEL):
        self.llm = ChatOpenAI(model=model, temperature=0.1)

        # Knowledge base (§5)
        self.retriever = InMemoryVectorStore.from_texts(
            policies, embedding=OpenAIEmbeddings(model=EMBED_MODEL)
        ).as_retriever(search_kwargs={"k": 2})
        self.qa_chain = build_qa_chain(self.retriever)

        # Agent over the tools, with memory (§4 + §8)
        self.agent = create_agent(
            model=model,
            tools=tools,
            system_prompt="You are a student support assistant. Use the tools when they help.",
            checkpointer=InMemorySaver(),
        )

        # Conversation history for general questions (§4)
        self.history = [SystemMessage(content="You are a supportive academic advisor. Keep replies short.")]

    def process_query(self, query, thread_id="support-session"):
        print("🔵 INPUT TO SUPPORT BOT:")
        print(f"Query: {query}")
        q = query.lower()

        if any(w in q for w in ["policy", "drop", "appeal", "probation", "requirement"]):
            print("🎯 Intent: Policy Question → Using Knowledge Base")
            print("\n🔵 SEARCHING KNOWLEDGE BASE...")
            result = self.qa_chain.invoke(query)

        elif any(w in q for w in ["student", "lookup", "info", "grade", "calculate"]):
            print("🎯 Intent: Student/Grade Query → Using Agent with Tools")
            print("\n🔵 CALLING AGENT...")
            out = self.agent.invoke(
                {"messages": [{"role": "user", "content": query}]},
                config={"configurable": {"thread_id": thread_id}},
            )
            result = out["messages"][-1].content

        else:
            print("🎯 Intent: General Support → Using Conversation Memory")
            print("\n🔵 CALLING GPT WITH MEMORY...")
            self.history.append(HumanMessage(content=query))
            reply = self.llm.invoke(self.history)
            self.history.append(reply)
            result = reply.content

        print("\n🤖 SUPPORT BOT RESPONSE:")
        print(result)
        return result


test_queries = [
    "What's the course drop policy?",
    "Look up student STU001 and calculate grade for 92 out of 100 points",
    "I'm feeling overwhelmed with my coursework. Any study tips?",
]

if HAS_KEY:
    print("🚀 Initializing Complete Educational Support Bot...")
    support_bot = EducationalSupportBot(educational_policies, student_tools)

    for q in test_queries:
        print(f"\n{'='*60}")
        print("TESTING COMPLETE SUPPORT BOT")
        print(f"{'='*60}")
        support_bot.process_query(q)

    print(f"\n{'='*60}")
    print("✅ LangChain Tutorial Complete!")
    print("You've learned: Models → Prompts → Chains → Memory → Retrieval → Tools → Agents → Skills → Integration")
    print(f"{'='*60}")
else:
    skipped(f"EducationalSupportBot routing {len(test_queries)} queries to RAG / agent / memory")

🚀 Initializing Complete Educational Support Bot...

TESTING COMPLETE SUPPORT BOT
🔵 INPUT TO SUPPORT BOT:
Query: What's the course drop policy?
🎯 Intent: Policy Question → Using Knowledge Base

🔵 SEARCHING KNOWLEDGE BASE...

🤖 SUPPORT BOT RESPONSE:
Students can drop courses within 2 weeks for a full refund. Partial refunds are available until week 4.

TESTING COMPLETE SUPPORT BOT
🔵 INPUT TO SUPPORT BOT:
Query: Look up student STU001 and calculate grade for 92 out of 100 points
🎯 Intent: Student/Grade Query → Using Agent with Tools

🔵 CALLING AGENT...

🤖 SUPPORT BOT RESPONSE:
Student STU001: Alice Johnson  
- Email: alice.johnson@university.edu  
- Major: Computer Science  
- GPA: 3.7  
- Current courses: CS101, MATH201, PHYS101  
- Support level: Premium

Grade for 92 out of 100: 92.0% (A)

TESTING COMPLETE SUPPORT BOT
🔵 INPUT TO SUPPORT BOT:
Query: I'm feeling overwhelmed with my coursework. Any study tips?
🎯 Intent: General Support → Using Conversation Memory

🔵 CALLING GPT WITH MEMOR

## Recap

| Component | How | Use it for |
|---|---|---|
| Models | `ChatOpenAI(model=...)` + `.invoke()` | any LLM call |
| Prompts | `ChatPromptTemplate.from_messages([...])` | reusable, parameterised instructions |
| Chains | `prompt \| llm \| parser` (LCEL) | fixed pipelines |
| Memory | carry the message list, or a LangGraph checkpointer | multi-turn conversations |
| RAG | `retriever \| format_docs` piped into the prompt | answers grounded in your documents |
| Tools | `@tool` | giving the model abilities |
| Agents | `create_agent` | dynamic, multi-step tasks |
| Skills | prompt + related tools, bundled (`SKILL.md`) | reusable, focused capabilities |
| Harness | `create_agent` or a custom LangGraph loop | running the model + tool loop |

**Next:** build something real for your capstone — start with a chain, add retrieval, then reach for an agent only when the steps genuinely depend on each other. Custom harnesses, observability and prompt optimization come in **Week 12 (Advanced GenAI)**.